In [1]:
%pip install -e ..

Obtaining file:///Users/pratikeliasjacob/Documents/Projects/simple-transformer
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for simple-transformer (pyproject.toml) ... done
  Created wheel for simple-transformer: filename=simple_transformer-0.1.0-0.editable-py3-none-any.whl size=2107 sha256=821ee68f7d3f57e9b7ad50b672aacb9db8d7f9783a9c2a43380d9fd1ef07d609
  Stored in directory: /private/var/folders/qn/79klx_zs0m1fqrh1q4c5ql_00000gn/T/pip-ephem-wheel-cache-c62et7th/wheels/de/d4/87/c813e0cb6465938cfd1c4350e8aa954bcfb9b50edc775763b4
Successfully built simple-transformer
  Attempting uninstall: simple-transformer
    Found existing installation: simple-transformer 0.1.0
    Uninstalling simple-transformer-0.1.0:
      Successfully uninstalled simple-transformer-0.1.0
Note: you may need to restart the kernel to 

In [2]:
import torch

from simple_transformer.config import small_addition_config
from simple_transformer.data import AdditionTokenizer, make_addition_dataloader
from simple_transformer.model import SimpleTransformerLM, count_parameters

In [3]:
print(torch.__version__)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.12.0
False


In [4]:
custom_examples = [
    "1+2=",
    "7+8=",
    "12+3=",
    "45+67=",
    "100+23=",
    "321+456=",
    "999+1=",
    "123+876=",
]

for example in custom_examples:
    print(example)

1+2=
7+8=
12+3=
45+67=
100+23=
321+456=
999+1=
123+876=


In [5]:
loader, tokenizer = make_addition_dataloader(
    num_examples=1000,
    max_digits=3,
    batch_size=4,
    seed=42,
)

batch = next(iter(loader))
print(batch)

{'input_ids': tensor([[ 6,  9,  5, 12,  6,  4,  5, 13, 10, 11,  8,  1],
        [ 4,  7,  6, 12,  4,  9,  5, 13,  7,  4,  9,  1],
        [ 6, 11,  7, 12,  4, 10,  5, 13,  9,  9, 10,  1],
        [10, 10,  9, 12,  6,  9,  4, 13,  3,  5,  7, 11]]), 'labels': tensor([[   9,    5,   12,    6,    4,    5,   13,   10,   11,    8,    1, -100],
        [   7,    6,   12,    4,    9,    5,   13,    7,    4,    9,    1, -100],
        [  11,    7,   12,    4,   10,    5,   13,    9,    9,   10,    1, -100],
        [  10,    9,   12,    6,    9,    4,   13,    3,    5,    7,   11,    1]])}


In [6]:
tokenizer = AdditionTokenizer()
config = small_addition_config(max_digits=3)
model = SimpleTransformerLM(config)
model.eval()

prompt_ids = [tokenizer.encode(example) for example in custom_examples]
generated_ids = model.generate_batch(
    prompt_ids,
    eos_token_id=tokenizer.eos_token_id,
)

print(f"parameters: {count_parameters(model):,}")
for prompt, output_ids in zip(custom_examples, generated_ids):
    print(f"{prompt} -> {tokenizer.decode(output_ids)}")

parameters: 1,068,160
1+2= -> 1+2==========
7+8= -> 7+8===2222222
12+3= -> 12+3=========
45+67= -> 45+67========
100+23= -> 100+23=======
321+456= -> 321+456======
999+1= -> 999+1==444444
123+876= -> 123+876======
